<h2>2026 NZF Impacts on Caribbean - 'trade_activity_mapping_v0.1.ipynb'</h2>

The initial EU ETS script looked at what EU ETS costs were likely to be for African states. This notebook goes one further and introduces NZF impact costs, using Scenario 24 of DNV's CIA Task 2 work, specifically reverse engineering policy impact onto the 4th IMO GHG Study dataset using DNV-derived Cost projections per unit of maritime transport supply (US$/dwt.nm). 

Forked from '/Users/apple/repos/PubEnv/2026_eu_ets_africa/2026_eu_ets_africa_v0.1.ipynb' on 24th March 2026.

In [1]:
# Import packages
import os
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None

<u><h2>Reading in Voyage Activity Dataset...</h2></u>

In [2]:
voys_nzf_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}
voys_nzf = pd.read_csv("/Users/apple/repos/CaribbEnv/2026/activity/activity_v0.1.csv", dtype=voys_nzf_d)
voys_nzf.head(2)

,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,Vessel Type,size_range_standardized,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,_voys_vess_m,dwtnm,bau_usd_23,bau_usd_30,bau_usd_40,bau_usd_50,s24_usd_23,s24_usd_30,s24_usd_40,s24_usd_50,ets_coverage,ets_23_usd,ets_30_usd
0,0,9114622,374609000,VNCPH,704,VNM,VN,Viet Nam,Cam Pha,CNLUH,156,CHN,CN,China,Luhuashan,2018-01-07 06:00:00,2018-01-12 23:00:00,137.0,310.0,447.0,1218.768412,44.583881,29.592668,74.176549,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.792272,1.189625,138.834205,92.151567,230.985772,142,Asia,035,South-eastern Asia,low_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.981897,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,5.806091e+07,159128.795463,191097.374915,216464.893320,228787.351725,159128.795463,191097.374915,216464.893320,228787.351725,0.0,0.0,0.0
1,1,9114622,374609000,CNLUH,156,CHN,CN,China,Luhuashan,CNCGS,156,CHN,CN,China,Changshu,2018-01-25 21:00:00,2018-01-28 06:00:00,57.0,497.0,554.0,155.368145,8.820822,47.640536,56.461358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.354597,1.915150,27.468039,148.352630,175.820669,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,2.269747,Bulk carrier,35000-59999,dwt,Panama,1996.0,26449.0,47639.0,NaN,NaN,NaN,NaN,SSD,1,both,7.401583e+06,20285.679803,24361.022450,27594.864284,29165.726710,20285.679803,24361.022450,27594.864284,29165.726710,0.0,0.0,0.0


# Introduce Trade-and-Transport Data to Represent International Bilateral Trade Flows

Considerations:

<ul>
    <li>All 'voys_foc_af' countries in TandT's 'Origin' field.</li>
    <li>'voys_foc_af' countries not in TandT's 'Destination' field include: Liberia, Guinea-Bissau, South Africa, Equatorial Guinea, Gabon, Djibouti, Saint Helena, Ascension and Tristan da Cunha.</li>
</ul>

In [3]:
print("Reading in the 'clean' TandT dataset representing the following aggregation: Origin.str.len() == 3 | Destination.str.len() == 3 | Product == TOTAL | TransportMode.isin(['10', '21', '31', '32', '80', '99', 'Ou'])\n")
TandT_dtypes = {"Origin":"str", "Origin Label":"str", "Destination":"str", "Destination Label":"str", "Product":"str", "TransportMode":"str"}
TandT_clean = pd.read_csv("/Users/apple/repos/datasets/unctad/trade_and_transport/data/US_TransportCosts_20182018_CLEAN.csv", dtype=TandT_dtypes)
display(TandT_clean.head(2))

print("\nSubsetting for HS2 codes only.")
TandT_by_mode = TandT_clean[(TandT_clean.Product.str.len() == 2)].reset_index().rename(columns={"index":"trade_idx"})
print("Total FOB value in Bilateral TandT dataset across all HS2s and Transport Modes: ${0}tn.".format(np.round(TandT_by_mode["FOB value (US$)"].sum() / 1e12, 1)))
print("Total weight in Bilateral TandT dataset across all HS2s and Transport Modes: {0}tn kg.".format(np.round(TandT_by_mode["Kilograms"].sum() / 1e12, 1)))
display(TandT_by_mode.head(2))

print("\nSubsetting for Seaborne trade only.")
TandT_sea = TandT_by_mode[(TandT_by_mode["TransportMode Label"] == "Sea")].drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
print("Total FOB value in Bilateral TandT dataset across all HS2s by Sea: ${0}tn.".format(np.round(TandT_sea["FOB value (US$)"].sum() / 1e12, 1)))
print("Total weight in Bilateral TandT dataset across all HS2s by Sea: {0}tn kg.".format(np.round(TandT_sea["Kilograms"].sum() / 1e12, 1)))
display(TandT_sea.head(2))

Reading in the 'clean' TandT dataset representing the following aggregation: Origin.str.len() == 3 | Destination.str.len() == 3 | Product == TOTAL | TransportMode.isin(['10', '21', '31', '32', '80', '99', 'Ou'])



,Origin,Origin Label,Destination,Destination Label,Product,TransportMode,TransportMode Label,FOB value (US$),Kilograms
0,004,Afghanistan,004,Afghanistan,TOTAL,80,Multimodal adjustment,NaN,NaN
1,004,Afghanistan,004,Afghanistan,01,80,Multimodal adjustment,NaN,NaN



Subsetting for HS2 codes only.
Total FOB value in Bilateral TandT dataset across all HS2s and Transport Modes: $16.5tn.
Total weight in Bilateral TandT dataset across all HS2s and Transport Modes: 12.7tn kg.


,trade_idx,Origin,Origin Label,Destination,Destination Label,Product,TransportMode,TransportMode Label,FOB value (US$),Kilograms
0,1,004,Afghanistan,004,Afghanistan,01,80,Multimodal adjustment,NaN,NaN
1,8,004,Afghanistan,004,Afghanistan,02,80,Multimodal adjustment,NaN,NaN



Subsetting for Seaborne trade only.
Total FOB value in Bilateral TandT dataset across all HS2s by Sea: $6.5tn.
Total weight in Bilateral TandT dataset across all HS2s by Sea: 6.3tn kg.


,trade_idx,Origin,Origin Label,Destination,Destination Label,Product,TransportMode,TransportMode Label,FOB value (US$),Kilograms
0,0,004,Afghanistan,012,Algeria,08,21,Sea,0.000,0.000
1,1,004,Afghanistan,012,Algeria,13,21,Sea,101.048,72.711


## Check ISO Code Consistency between the two Datasets

In [4]:
print("\nValues from the 'source_iso_code' field of 'voys_nzf' not found in 'Origin' field of the TandT database:\n")
voys_source_iso_not_in_TandT_sea_origin = [row.source_country for idx, row in voys_nzf[["source_iso_code", "source_country"]].drop_duplicates().iterrows() if row.source_iso_code not in TandT_sea.Origin.unique()]
print("{0} Countries in total, including: \n".format(len(voys_source_iso_not_in_TandT_sea_origin)))
[print(i) for i in voys_source_iso_not_in_TandT_sea_origin]
print()

print("\nValues from the 'dest_iso_code' field of 'voys_nzf' not found in 'Destination' field of the TandT database:\n")
voys_dest_iso_not_in_TandT_sea_dest = [row.dest_country for idx, row in voys_nzf[["dest_iso_code", "dest_country"]].drop_duplicates().iterrows() if row.dest_iso_code not in TandT_sea.Destination.unique()]
print("{0} Countries in total, including: \n".format(len(voys_dest_iso_not_in_TandT_sea_dest)))
[print(i) for i in voys_dest_iso_not_in_TandT_sea_dest]
print()


print("\nValues from the 'Origin' field of the TandT databasenot found in the 'source_iso_code' field of 'voys_nzf':\n")
TandT_sea_origin_not_in_voys_source = [row["Origin Label"] for idx, row in TandT_sea[["Origin", "Origin Label"]].drop_duplicates().iterrows() if row.Origin not in voys_nzf.source_iso_code.unique()]
print("{0} Countries in total, including: \n".format(len(TandT_sea_origin_not_in_voys_source)))
[print(i) for i in TandT_sea_origin_not_in_voys_source]
print()

print("\nValues from the 'Destination' field of the TandT database not found in the 'dest_iso_code' field of 'voys_nzf':\n")
TandT_sea_dest_not_in_voys_dest = [row["Destination Label"] for idx, row in TandT_sea[["Destination", "Destination Label"]].drop_duplicates().iterrows() if row.Destination not in voys_nzf.dest_iso_code.unique()]
print("{0} Countries in total, including: \n".format(len(TandT_sea_dest_not_in_voys_dest)))
[print(i) for i in TandT_sea_dest_not_in_voys_dest]
print()


Values from the 'source_iso_code' field of 'voys_nzf' not found in 'Origin' field of the TandT database:

25 Countries in total, including: 

Norway
France
United Kingdom of Great Britain and Northern Ireland
United States of America
Martinique
Guadeloupe
Congo
Virgin Islands (U.S.)
Puerto Rico
Antarctica
Svalbard and Jan Mayen
Réunion
Mayotte
French Guiana
Saint Martin (French part)
Monaco
Guernsey
Cocos (Keeling) Islands
Jersey
Isle of Man
Pitcairn
South Georgia and the South Sandwich Islands
Åland Islands
Wallis and Futuna
Switzerland


Values from the 'dest_iso_code' field of 'voys_nzf' not found in 'Destination' field of the TandT database:

74 Countries in total, including: 

Taiwan, Province of China
Bangladesh
Norway
Liberia
France
Gibraltar
South Africa
United Kingdom of Great Britain and Northern Ireland
Papua New Guinea
United States of America
Mexico
Venezuela (Bolivarian Republic of)
Martinique
Iraq
Djibouti
Vanuatu
New Caledonia
Canada
Guinea-Bissau
Cuba
Guadeloupe
Domin

In [5]:
# Provide a clean iso_code field where current value is wrong based on the above.
TandT_sea.loc[TandT_sea.Origin == "251", "Origin"] = "250" # France
TandT_sea.loc[TandT_sea.Destination == "251", "Destination"] = "250"
TandT_sea.loc[TandT_sea.Origin == "579", "Origin"] = "578" # Norway
TandT_sea.loc[TandT_sea.Destination == "579", "Destination"] = "578"
TandT_sea.loc[TandT_sea.Origin == "757", "Origin"] = "756" # Switzerland
TandT_sea.loc[TandT_sea.Destination == "757", "Destination"] = "756"
TandT_sea.loc[TandT_sea.Origin == "842", "Origin"] = "840" # USA
TandT_sea.loc[TandT_sea.Destination == "842", "Destination"] = "840"
TandT_sea.loc[TandT_sea.Origin == "926", "Origin"] = "826" # UK
TandT_sea.loc[TandT_sea.Destination == "926", "Destination"] = "826"
TandT_sea.loc[TandT_sea.Origin == "080", "Origin"] = "010" # British Overseas Antarctic Territory
TandT_sea.loc[TandT_sea.Destination == "080", "Destination"] = "010"
display(TandT_sea.iloc[:2])

,trade_idx,Origin,Origin Label,Destination,Destination Label,Product,TransportMode,TransportMode Label,FOB value (US$),Kilograms
0,0,004,Afghanistan,012,Algeria,08,21,Sea,0.000,0.000
1,1,004,Afghanistan,012,Algeria,13,21,Sea,101.048,72.711


In [6]:
print("\nValues from the 'source_iso_code' field of 'voys_nzf' not found in 'Origin' field of the TandT database:\n")
voys_source_iso_not_in_TandT_sea_origin = pd.Series([row.source_country for idx, row in voys_nzf[["source_iso_code", "source_country"]].drop_duplicates().iterrows() if row.source_iso_code not in TandT_sea.Origin.unique()]).sort_values()
print("{0} Countries in total, including: \n".format(len(voys_source_iso_not_in_TandT_sea_origin)))
[print(i) for i in voys_source_iso_not_in_TandT_sea_origin]
print()

print("\nValues from the 'dest_iso_code' field of 'voys_nzf' not found in 'Destination' field of the TandT database:\n")
voys_dest_iso_not_in_TandT_sea_dest = pd.Series([row.dest_country for idx, row in voys_nzf[["dest_iso_code", "dest_country"]].drop_duplicates().iterrows() if row.dest_iso_code not in TandT_sea.Destination.unique()]).sort_values()
print("{0} Countries in total, including: \n".format(len(voys_dest_iso_not_in_TandT_sea_dest)))
[print(i) for i in voys_dest_iso_not_in_TandT_sea_dest]
print()


print("\nValues from the 'Origin' field of the TandT databasenot found in the 'source_iso_code' field of 'voys_nzf':\n")
TandT_sea_origin_not_in_voys_source = pd.Series([row["Origin Label"] for idx, row in TandT_sea[["Origin", "Origin Label"]].drop_duplicates().iterrows() if row.Origin not in voys_nzf.source_iso_code.unique()]).sort_values()
print("{0} Countries in total, including: \n".format(len(TandT_sea_origin_not_in_voys_source)))
[print(i) for i in TandT_sea_origin_not_in_voys_source]
print()

print("\nValues from the 'Destination' field of the TandT database not found in the 'dest_iso_code' field of 'voys_nzf':\n")
TandT_sea_dest_not_in_voys_dest = pd.Series([row["Destination Label"] for idx, row in TandT_sea[["Destination", "Destination Label"]].drop_duplicates().iterrows() if row.Destination not in voys_nzf.dest_iso_code.unique()]).sort_values()
print("{0} Countries in total, including: \n".format(len(TandT_sea_dest_not_in_voys_dest)))
[print(i) for i in TandT_sea_dest_not_in_voys_dest]
print()


Values from the 'source_iso_code' field of 'voys_nzf' not found in 'Origin' field of the TandT database:

20 Countries in total, including: 

Antarctica
Cocos (Keeling) Islands
Congo
French Guiana
Guadeloupe
Guernsey
Isle of Man
Jersey
Martinique
Mayotte
Monaco
Pitcairn
Puerto Rico
Réunion
Saint Martin (French part)
South Georgia and the South Sandwich Islands
Svalbard and Jan Mayen
Virgin Islands (U.S.)
Wallis and Futuna
Åland Islands


Values from the 'dest_iso_code' field of 'voys_nzf' not found in 'Destination' field of the TandT database:

69 Countries in total, including: 

American Samoa
Anguilla
Antarctica
Bangladesh
Bonaire, Sint Eustatius and Saba
Canada
Christmas Island
Cocos (Keeling) Islands
Congo
Cook Islands
Cuba
Curaçao
Djibouti
Dominica
Equatorial Guinea
Eritrea
Falkland Islands (Malvinas)
Faroe Islands
French Guiana
Gabon
Gibraltar
Guadeloupe
Guam
Guernsey
Guinea-Bissau
Haiti
Iraq
Isle of Man
Jersey
Korea (Democratic People's Republic of)
Liberia
Marshall Islands
Mar

<u><h1>Format Output</h1></u>

In [7]:
TandT_r = {
    "Origin":"source_iso_code", "Origin Label":"source_country", "Destination":"dest_iso_code", "Destination Label":"dest_country", 
    "Product":"HS2", "TransportMode":"mode_code", "TransportMode Label":"mode_label", 
    "FOB value (US$)":"val_usd", "Kilograms":"vol_kg"
}

print("\nSpecifying Export-side Trade Flows.\n")
TandT_ex = TandT_sea.rename(columns=TandT_r).drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
print("Length of 'TandT_ex' dataset by Sea: {0}".format(TandT_ex.shape[0]))
print("Number of source countries in 'TandT_ex' dataset: {0}.".format(len(pd.Series(TandT_ex.source_country.unique()).sort_values().values)))
print("Number of destination countries in 'TandT_ex' dataset: {0}.".format(len(pd.Series(TandT_ex.dest_country.unique()).sort_values().values)))
print("\nTotal value in dataset: ${0}bn".format(np.round(TandT_ex.val_usd.sum() / 1e9, 1)))
print("Total weight in dataset: {0}bn kg".format(np.round(TandT_ex.vol_kg.sum() / 1e9, 1)))
TandT_ex.to_csv("/Users/apple/repos/CaribbEnv/2026/trade/trade_ex_v0.1.csv", index=False)
display(TandT_ex.head(2))


Specifying Export-side Trade Flows.

Length of 'TandT_ex' dataset by Sea: 633319
Number of source countries in 'TandT_ex' dataset: 222.
Number of destination countries in 'TandT_ex' dataset: 169.

Total value in dataset: $6537.5bn
Total weight in dataset: 6331.6bn kg


,trade_idx,source_iso_code,source_country,dest_iso_code,dest_country,HS2,mode_code,mode_label,val_usd,vol_kg
0,0,004,Afghanistan,012,Algeria,08,21,Sea,0.000,0.000
1,1,004,Afghanistan,012,Algeria,13,21,Sea,101.048,72.711


In [8]:
print("\nSpecifying Import-side Trade Flows.\n")
TandT_im = TandT_sea.rename(columns=TandT_r).drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
print("Length of 'TandT_im' dataset by Sea: {0}".format(TandT_im.shape[0]))
print("Number of source countries in 'TandT_im' dataset: {0}.".format(len(pd.Series(TandT_im.source_country.unique()).sort_values().values)))
print("Number of destination countries in 'TandT_im' dataset: {0}.".format(len(pd.Series(TandT_im.dest_country.unique()).sort_values().values)))
print("\nTotal value in dataset: ${0}bn".format(np.round(TandT_im.val_usd.sum() / 1e9, 1)))
print("Total weight in dataset: {0}bn kg".format(np.round(TandT_im.vol_kg.sum() / 1e9, 1)))
TandT_im.to_csv("/Users/apple/repos/CaribbEnv/2026/trade/trade_im_v0.1.csv", index=False)
display(TandT_im.head(2))


Specifying Import-side Trade Flows.

Length of 'TandT_im' dataset by Sea: 633319
Number of source countries in 'TandT_im' dataset: 222.
Number of destination countries in 'TandT_im' dataset: 169.

Total value in dataset: $6537.5bn
Total weight in dataset: 6331.6bn kg


,trade_idx,source_iso_code,source_country,dest_iso_code,dest_country,HS2,mode_code,mode_label,val_usd,vol_kg
0,0,004,Afghanistan,012,Algeria,08,21,Sea,0.000,0.000
1,1,004,Afghanistan,012,Algeria,13,21,Sea,101.048,72.711
